# Projeto Extensionista - Dashboard de Enchentes em Porto Alegre - INCT SiMAI - FASE 2
**Nomes:** Amanda Flores Domeneghini, Milena Fontana Bregalda, Mariana Nunes dos Santos, Karen Cristiane Nascimento


**Disciplina**: Coleta, Preparação e Análise de Dados


**Turma**: 310/110

**Professora**: Katherine Bianchini Esper


**Data**: 09/11/2026

Dependências necessárias:

In [6]:
!pip install requests
!pip install pandas

Script:

In [ ]:
import requests
import json
import pandas as pd # Talvez vai ser usado depois, por enquanto so JSON

# Obs.: por enquanto, pandas ainda nao eh utilizada (pode ser que venha a ser),
# e o arquivo eh gerado na mesma pasta em que o script esta.

# Link da primeira API (Cotas de Inundacao NAO RODAR TA COM PROBLEMA):
url = "https://agis-smamus.portoalegre.rs.gov.br/server/rest/services/01_PUBLICACOES/cotas_inundacao_300_535/FeatureServer/0/query"

# Primeiro, consulta os IDs de todos os registros:
parametros_ids = {
    "where": "1=1",
    "returnIdsOnly": "true",
    "f": "json"
}

resposta = requests.get(url, params=parametros_ids)
dados_ids = resposta.json()

if "error" in dados_ids:
    print("Erro ao buscar IDs:", dados_ids["error"])

else:
    ids = dados_ids["objectIds"]
    print("Quantidade de registros encontrados:", len(ids))

    # Busca as geometrias em lotes de 5 registros:
    todas_features = []

    for i in range(0, len(ids), 5):
        lote = ids[i:i+5]

        parametros = {
            "objectIds": ",".join(map(str, lote)),
            "outFields": "*",
            "returnGeometry": "true",
            "f": "json"
        }

        resposta = requests.get(url, params=parametros)
        dados = resposta.json()

        if "error" in dados:
            print("Erro ao buscar o lote:", dados["error"])
        else:
            todas_features.extend(dados.get("features", []))

    # Monta o resultado final:
    resultado = {
        "geometryType": "esriGeometryPolygon",
        "features": todas_features
    }

    # Salva os dados em arquivo JSON:
    with open("dadosapi_cotasdeinundacao.json", "w", encoding="utf-8") as f:
        json.dump(resultado, f, ensure_ascii=False, indent=4)

    print("Arquivo JSON salvo com sucesso!")
    print("Quantidade de polígonos baixados:", len(todas_features))

Quantidade de registros encontrados: 25


abaixo tentativas pra fazer funcionar, aparentemente ele pega dados muito grandes por isso da erro e crasha
(o dado completo nao terminou e o json tava dando 200mb), procurando maneiras de ver se funciona e simplificar os dados q pega, pegando so os q respondem a pergunta:

In [8]:
import requests

url = "https://gis-smamus.portoalegre.rs.gov.br/server/rest/services/01_PUBLICACOES/cotas_inundacao_300_535/FeatureServer/0/query"

parametros = {
    "where": "1=1",
    "returnCountOnly": "true",
    "f": "json"
}

resposta = requests.get(url, params=parametros)

print("Status HTTP:", resposta.status_code)
print("Resposta:", resposta.json())

Status HTTP: 200
Resposta: {'count': 25}


In [9]:
import requests

url = "https://gis-smamus.portoalegre.rs.gov.br/server/rest/services/01_PUBLICACOES/cotas_inundacao_300_535/FeatureServer/0/query"

parametros = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "false",
    "f": "json"
}

resposta = requests.get(url, params=parametros)

print("Status HTTP:", resposta.status_code)

dados = resposta.json()

if "error" in dados:
    print("Erro da API:", dados["error"])
else:
    print("Quantidade de registros:", len(dados.get("features", [])))
    print("Exemplo de registro:", dados["features"][0])

Status HTTP: 200
Quantidade de registros: 25
Exemplo de registro: {'attributes': {'objectid': 1, 'contourmax': 300.0, 'area_ha': 33335.5391246, 'Shape__Area': 0.03121776864390721, 'Shape__Length': 9.140178949917088}}


In [10]:
import requests

url = "https://gis-smamus.portoalegre.rs.gov.br/server/rest/services/01_PUBLICACOES/cotas_inundacao_300_535/FeatureServer/0/query"

parametros = {
    "objectIds": "1",
    "outFields": "objectid",
    "returnGeometry": "true",
    "f": "json"
}

resposta = requests.get(url, params=parametros)
dados = resposta.json()

print("Status HTTP:", resposta.status_code)

if "error" in dados:
    print("Erro da API:", dados["error"])
else:
    print("Quantidade de polígonos:", len(dados["features"]))
    print("Tipo de geometria:", dados.get("geometryType"))
    print("Geometria:", dados["features"][0].get("geometry"))

Status HTTP: 200
Quantidade de polígonos: 1
Tipo de geometria: esriGeometryPolygon
Geometria: {'rings': [[[-51.25802435199995, -30.112777687999937], [-51.25791586599996, -30.11255598799994], [-51.25781620499998, -30.11253566199997], [-51.25770417599995, -30.112680590999958], [-51.25766666199996, -30.112918139999977], [-51.257609686999956, -30.11292108999993], [-51.257523970999955, -30.11301965499996], [-51.25733895999997, -30.113489570999945], [-51.25735196599993, -30.113568330999954], [-51.25740385599994, -30.113572329999954], [-51.257507156999964, -30.113395126999933], [-51.257611349999934, -30.113562184999978], [-51.257662901999936, -30.11343562999997], [-51.25773428599996, -30.11345481999996], [-51.25770497099995, -30.11350867699997], [-51.25777976899997, -30.11348855999995], [-51.25767894299997, -30.113693814999976], [-51.257547425999974, -30.113624990999938], [-51.257412329999966, -30.113680973999976], [-51.257414231999974, -30.113726073999942], [-51.25759523499994, -30.113861031

In [1]:
import requests

url = "https://gis-smamus.portoalegre.rs.gov.br/server/rest/services/01_PUBLICACOES/cotas_inundacao_300_535/FeatureServer/0/query"

parametros = {
    "where": "1=1",
    "outFields": "objectid,contourmax,area_ha",
    "returnGeometry": "false",
    "f": "json"
}

resposta = requests.get(url, params=parametros, timeout=30)
dados = resposta.json()

if "error" in dados:
    print("Erro:", dados["error"])
else:
    for registro in dados["features"]:
        print(registro["attributes"])

{'objectid': 1, 'contourmax': 300.0, 'area_ha': 33335.5391246}
{'objectid': 2, 'contourmax': 310.0, 'area_ha': 33712.3655149}
{'objectid': 3, 'contourmax': 320.0, 'area_ha': 34101.2527045}
{'objectid': 4, 'contourmax': 330.0, 'area_ha': 34427.9494234}
{'objectid': 5, 'contourmax': 340.0, 'area_ha': 34737.2939789}
{'objectid': 6, 'contourmax': 350.0, 'area_ha': 34991.7385174}
{'objectid': 7, 'contourmax': 360.0, 'area_ha': 35226.1115026}
{'objectid': 8, 'contourmax': 370.0, 'area_ha': 35448.5931684}
{'objectid': 9, 'contourmax': 380.0, 'area_ha': 35619.0033893}
{'objectid': 10, 'contourmax': 390.0, 'area_ha': 35795.0632228}
{'objectid': 11, 'contourmax': 400.0, 'area_ha': 35953.1185032}
{'objectid': 12, 'contourmax': 410.0, 'area_ha': 36098.7196816}
{'objectid': 13, 'contourmax': 420.0, 'area_ha': 36258.2666137}
{'objectid': 14, 'contourmax': 430.0, 'area_ha': 36411.4981826}
{'objectid': 15, 'contourmax': 440.0, 'area_ha': 36566.5629538}
{'objectid': 16, 'contourmax': 450.0, 'area_ha': 

In [4]:
import requests
import json

url = "https://gis-smamus.portoalegre.rs.gov.br/server/rest/services/01_PUBLICACOES/cotas_inundacao_300_535/FeatureServer/0/query"

parametros = {
    "where": "objectid=1",
    "outFields": "objectid,contourmax,area_ha",
    "returnGeometry": "true",
    "maxAllowableOffset": 20,
    "f": "json"
}

try:
    resposta = requests.get(url, params=parametros, timeout=120)
    dados = resposta.json()

    if "error" in dados:
        print("Erro:", dados["error"])
    else:
        geometria = dados["features"][0]["geometry"]

        print("Quantidade de polígonos:", len(dados["features"]))
        print("Quantidade de pontos:", sum(len(ring) for ring in geometria["rings"]))

        with open("teste_poligono_simplificado.json", "w", encoding="utf-8") as arquivo:
            json.dump(dados, arquivo, ensure_ascii=False)

        print("Arquivo salvo com sucesso!")

except requests.exceptions.Timeout:
    print("A API demorou mais de 120 segundos.")
except requests.exceptions.RequestException as erro:
    print("Erro na conexão:", erro)

A API demorou mais de 120 segundos.
